# Lekcja 09 - Wzorzec projektowy metapoznania


## Konfiguracja

Ten notatnik demonstruje wzorzec projektowy Metacognition z użyciem Microsoft Agent Framework.

**Wymagania wstępne:**
- Wdrożenie Azure OpenAI skonfigurowane za pomocą zmiennych środowiskowych
- Azure CLI uwierzytelnione (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Czym jest metapoznanie?

Metapoznanie to **myślenie o myśleniu**. W kontekście agentów AI oznacza to tworzenie agentów, którzy potrafią:

- **Samorefleksja** nad własnymi wynikami i procesem rozumowania
- **Wykrywać błędy** i reagować naprawczo zamiast zawodzić po cichu
- **Ocenić**, czy ich odpowiedzi są kompletne i pomocne
- **Dostosować** swoją strategię, gdy początkowe podejście nie działa (np. przejście na system zapasowy)

Agent metapoznawczy nie tylko odpowiada na pytania — monitoruje własne działanie i dostosowuje się w locie.


## Narzędzia główne i zapasowe

Powszechnym wzorcem metapoznawczym jest **strategia awaryjna**. Agent najpierw próbuje narzędzia głównego; jeśli ono zawiedzie (np. błąd 404), agent rozpoznaje awarię i w przejrzysty sposób przełącza się na narzędzie zapasowe.

To odzwierciedla systemy w świecie rzeczywistym, w których usługi główne mogą być niedostępne, a agenci muszą samodzielnie zdiagnozować problem, zanim wybiorą alternatywną ścieżkę.

Poniżej definiujemy dwa narzędzia do wyszukiwania lotów:
- **Główne** — obsługuje Paryż, Tokio i Barcelonę
- **Zapasowe** — obsługuje Berlin, Sydney i Nowy Jork


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Agent samoreflektujący z odzyskiwaniem po błędach

Poniższy agent ma instrukcję, aby najpierw spróbować użyć głównego systemu lotu, rozpoznać awarie i w przejrzysty sposób przełączyć się na system zapasowy. Po każdej odpowiedzi krótko samoocenia, czy w pełni odpowiedział na pytanie użytkownika.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Wzorzec samooceny

Innym aspektem metapoznania jest **samoocena**: oddzielny agent (lub ten sam agent przy drugim przejściu) przegląda odpowiedź pod kątem kompletności, dokładności i przydatności.

Poniżej tworzymy agenta `ResponseEvaluator`, który ocenia odpowiedzi agenta podróży w trzech wymiarach.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Podsumowanie

W tej lekcji dowiedziałeś się, jak budować **agentów metapoznawczych** przy użyciu Microsoft Agent Framework:

- **Samorefleksja**: Agenci, którzy monitorują własne rozumowanie i przejrzyście komunikują, co się stało.
- **Odzyskiwanie po błędach za pomocą zapasowych rozwiązań**: Wzorzec narzędzia głównego + zapasowego, w którym agent wykrywa awarie (np. błędy 404) i automatycznie próbuje alternatywnego źródła.
- **Samoocena**: Oddzielny agent ewaluujący, który ocenia odpowiedzi pod kątem kompletności, dokładności i przydatności.

Te wzorce czynią agentów bardziej odpornymi, przejrzystymi i godnymi zaufania — to kluczowe cechy dla wdrożeń produkcyjnych.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Zastrzeżenie:
Niniejszy dokument został przetłumaczony przy użyciu usługi tłumaczeniowej opartej na sztucznej inteligencji [Co-op Translator](https://github.com/Azure/co-op-translator). Chociaż dokładamy starań, aby tłumaczenia były jak najdokładniejsze, prosimy pamiętać, że automatyczne przekłady mogą zawierać błędy lub nieścisłości. Oryginalny dokument w jego języku źródłowym należy uważać za źródło wiążące. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z korzystania z tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
